# Train a tiny CNN in PyTorch

Each image is 12 by 12 pixels of faint noise with one bright bar across it: a column (class 0) or a
row (class 1). In the training images the bar always sits in column or row 2 to 6. You will write
convolution and max-pooling by hand and check them against PyTorch, build two small CNNs and test
how their feature maps respond when the input shifts, and then find out which networks still
recognize a bar in a position no training image used.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It loads the libraries and the data every part uses.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F


def make_bars(positions, n, seed, size=12):
    """n noisy size x size images with one bright bar each.

    Class 0 has a vertical bar (a bright column), class 1 a horizontal bar (a bright row).
    The column or row is drawn from `positions`. Returns X of shape (n, 1, size, size) and y of shape (n,).
    """
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 2, (n,), generator=g)
    pos = torch.tensor(list(positions))[torch.randint(0, len(positions), (n,), generator=g)]
    X = 0.1 * torch.randn(n, 1, size, size, generator=g)
    idx = torch.arange(size)
    vertical = (idx[None, None, :] == pos[:, None, None]).expand(n, size, size)
    horizontal = (idx[None, :, None] == pos[:, None, None]).expand(n, size, size)
    X[:, 0] += 1.5 * torch.where(y[:, None, None] == 0, vertical, horizontal)
    return X, y


# Training bars sit in columns or rows 2 to 6 only. The second test set uses 7 to 9,
# positions no training image ever had.
X_train, y_train = make_bars(range(2, 7), 256, seed=0)
X_seen, y_seen = make_bars(range(2, 7), 200, seed=1)
X_new, y_new = make_bars(range(7, 10), 200, seed=2)

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, X, y, i in zip(axes, [X_train] * 3 + [X_new] * 3, [y_train] * 3 + [y_new] * 3, [0, 1, 2, 0, 1, 2]):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(("vertical" if y[i] == 0 else "horizontal") + (" (new)" if X is X_new else ""), fontsize=8)
    ax.axis("off")
plt.show()

## Part 1 (coding task): Convolution and pooling by hand

A 2-D convolution layer (strictly, a cross-correlation, which is what PyTorch computes) with weight $W$ of shape `(C_out, C_in, k, k)` and bias $\mathbf{b}$ maps an input $x$ of shape `(N, C_in, H, W)` to

$$
\text{out}[n, o, i, j] = b_o + \sum_{c}\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} W[o, c, u, v]\; x_{\text{pad}}[n, c, i + u, j + v],
$$

where $x_{\text{pad}}$ is $x$ with `padding` zeros added on every side.

- `conv_output_size(n, k, padding, stride=1)` returns the output height for input height `n`: $\lfloor (n + 2p - k)/s \rfloor + 1$.
- `conv2d_manual(x, weight, bias, padding=0)` computes the formula above with stride 1. Loop over the $k^2$ kernel offsets $(u, v)$ only: for each one, take the shifted slice of $x_{\text{pad}}$ and contract its channel axis with `weight[:, :, u, v]` (`torch.einsum` does this in one call). Do not loop over images, channels or output pixels.
- `max_pool_manual(x, k)` takes the maximum over non-overlapping `k` by `k` windows, for `H` and `W` divisible by `k`. Reshape `(N, C, H, W)` to `(N, C, H/k, k, W/k, k)` and reduce.

In [ ]:
def conv_output_size(n, k, padding, stride=1):
    # TODO
    raise NotImplementedError


def conv2d_manual(x, weight, bias, padding=0):
    # TODO
    raise NotImplementedError


def max_pool_manual(x, k):
    # TODO
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 3, 9, 7)
w = torch.randn(5, 3, 3, 3)
b = torch.randn(5)
for p in (0, 1, 2):
    mine = conv2d_manual(x, w, b, padding=p)
    ref = F.conv2d(x, w, b, padding=p)
    assert mine.shape == ref.shape, f"padding {p}: shape {tuple(mine.shape)}, expected {tuple(ref.shape)}"
    assert torch.allclose(mine, ref, atol=1e-5), f"padding {p}: values differ from F.conv2d"
    assert conv_output_size(9, 3, p) == ref.shape[2], "conv_output_size disagrees with F.conv2d"
assert conv_output_size(32, 5, 2, stride=2) == 16 and conv_output_size(28, 3, 0, stride=2) == 13, "check the stride"
x = torch.randn(2, 3, 8, 6)
assert torch.equal(max_pool_manual(x, 2), F.max_pool2d(x, 2)), "max_pool_manual differs from F.max_pool2d"
print("Part 1 passed")

## Part 2 (coding task): Two CNNs and the shift test

Build two small networks, both starting with `nn.Conv2d(1, 4, kernel_size=3, padding=1)` and a ReLU.

- `TinyCNN` then applies a 2 by 2 max-pool, flattens the `(4, 6, 6)` result into 144 numbers and maps them to 2 scores with `nn.Linear(144, 2)`.
- `GlobalCNN` instead takes the maximum of each of the 4 feature maps over **all** positions (global max-pooling) and maps the 4 numbers to 2 scores with `nn.Linear(4, 2)`.

Then write `shift_test(model, x, shift)`: roll the images `x` by `shift` pixels to the right with `torch.roll` along the width axis, and return `(conv_shifts, output_same)`. `conv_shifts` is `True` when the model's ReLU feature maps for the rolled images equal the feature maps of the original images rolled by the same amount, in the columns away from the left and right borders (columns 2 to 9). `output_same` is `True` when the model's scores for the original and rolled images agree to within $10^{-5}$.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        raise NotImplementedError

    def features(self, x):
        """The ReLU feature maps, shape (N, 4, 12, 12)."""
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class GlobalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        raise NotImplementedError

    def features(self, x):
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


def shift_test(model, x, shift):
    # TODO
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
tiny, glob = TinyCNN(), GlobalCNN()
assert sum(p.numel() for p in tiny.parameters()) == 330, "TinyCNN: (1*3*3 + 1)*4 + (144 + 1)*2 = 330 parameters"
assert sum(p.numel() for p in glob.parameters()) == 50, "GlobalCNN: 40 + (4 + 1)*2 = 50 parameters"
assert tiny(X_train[:3]).shape == (3, 2) and glob(X_train[:3]).shape == (3, 2), "two scores per image"

bar = torch.zeros(1, 1, 12, 12)
bar[0, 0, :, 4] = 1.5                       # a clean vertical bar in column 4
for model in (tiny, glob):
    conv_shifts, _ = shift_test(model, bar, 3)
    assert conv_shifts, f"{type(model).__name__}: the feature maps should shift with the input"
print("TinyCNN output unchanged by the shift:", shift_test(tiny, bar, 3)[1])
print("GlobalCNN output unchanged by the shift:", shift_test(glob, bar, 3)[1])
assert not shift_test(tiny, bar, 3)[1], "TinyCNN's linear layer sees where the features are"
assert shift_test(glob, bar, 3)[1], "global max-pooling should make the output ignore the shift"
print("Part 2 passed")

## Part 3 (coding task): Which networks recognize bars in new places?

Write `train(model, X, y, steps=100, lr=0.01)`: full-batch training with `nn.CrossEntropyLoss` and Adam for `steps` steps, returning the final loss as a number. Write `accuracy(model, X, y)`, the fraction of images whose higher score is the right class, as a number.

Then write `compare(builders, seeds=5)`. `builders` maps a name to a function that builds a fresh model. For each name and each seed in `range(seeds)`, call `torch.manual_seed(seed)`, build the model, train it on `X_train`, and record its accuracy on `X_seen` (bars at the training positions) and on `X_new`, whose bars sit at positions 7 to 9, where no training bar ever was. Return a dictionary from each name to `(seen_accuracies, new_accuracies)`, two lists with one number per seed. The check runs it on three networks: a dense network, `TinyCNN` and `GlobalCNN`.

In [ ]:
def train(model, X, y, steps=100, lr=0.01):
    # TODO
    raise NotImplementedError


def accuracy(model, X, y):
    # TODO
    raise NotImplementedError


def compare(builders, seeds=5):
    """name -> (accuracies on X_seen, accuracies on X_new), one entry per seed."""
    # TODO
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
probe = nn.Sequential(nn.Flatten(), nn.Linear(144, 16), nn.ReLU(), nn.Linear(16, 2))
before = accuracy(probe, X_seen, y_seen)
assert not isinstance(before, torch.Tensor), "accuracy should return a plain number, not a tensor: end with .item()"
assert 0.0 <= float(before) <= 1.0, f"accuracy {before} should lie between 0 and 1"
start_loss = nn.CrossEntropyLoss()(probe(X_train), y_train).item()
final = train(probe, X_train, y_train, steps=20)
assert not isinstance(final, torch.Tensor), "train should return the final loss as a plain number: end with .item()"
assert float(final) < start_loss, f"twenty steps of training should lower the loss, but it went from {start_loss:.3f} to {float(final):.3f}"

builders = {
    "dense": lambda: nn.Sequential(nn.Flatten(), nn.Linear(144, 16), nn.ReLU(), nn.Linear(16, 2)),
    "tiny": TinyCNN,
    "global": GlobalCNN,
}
results = compare(builders)
assert set(results) == {"dense", "tiny", "global"}, f"results should have one entry per builder, got {sorted(results)}"
assert all(len(s) == 5 and len(n) == 5 for s, n in results.values()), "5 runs per network, each with an accuracy on X_seen and one on X_new"
mean = {name: (sum(s) / 5, sum(n) / 5) for name, (s, n) in results.items()}
for name, (seen, new) in mean.items():
    print(f"{name:7s} seen positions {seen:.3f}   new positions {new:.3f}")
assert all(m[0] > 0.98 for m in mean.values()), "all three should be near perfect on the positions they trained on"
assert mean["global"][1] > 0.98, "global max-pooling should recognize bars at new positions"
assert mean["dense"][1] < 0.2, "the dense network should fail on new positions"
assert mean["tiny"][1] < 0.8, "TinyCNN's position-specific linear layer should not transfer well"
print("Part 3 passed")

## Solution

### Part 1: Convolution and pooling by hand

In [ ]:
def conv_output_size(n, k, padding, stride=1):
    return (n + 2 * padding - k) // stride + 1


def conv2d_manual(x, weight, bias, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, k, _ = weight.shape
    xp = F.pad(x, (padding, padding, padding, padding))
    H_out, W_out = conv_output_size(H, k, padding), conv_output_size(W, k, padding)
    out = torch.zeros(N, C_out, H_out, W_out)
    for u in range(k):
        for v in range(k):
            patch = xp[:, :, u:u + H_out, v:v + W_out]                 # (N, C_in, H_out, W_out)
            out += torch.einsum("nchw,oc->nohw", patch, weight[:, :, u, v])
    return out + bias[None, :, None, None]


def max_pool_manual(x, k):
    N, C, H, W = x.shape
    return x.reshape(N, C, H // k, k, W // k, k).amax(dim=(3, 5))

**Why it works.** Each output pixel is a dot product between the kernel and the $k \times k$ patch under it, and the same kernel is used at every position. Looping over the $k^2$ offsets instead of the output pixels turns the work into $k^2$ large tensor operations: offset $(u, v)$ contributes `weight[:, :, u, v]` times the input shifted by $(u, v)$, for every pixel at once. The reshape for pooling splits each spatial axis into (window index, position inside the window), so the maximum over the two inner axes is the maximum over each window.

### Part 2: Two CNNs and the shift test

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.fc = nn.Linear(4 * 6 * 6, 2)

    def features(self, x):
        """The ReLU feature maps, shape (N, 4, 12, 12)."""
        return F.relu(self.conv(x))

    def forward(self, x):
        return self.fc(F.max_pool2d(self.features(x), 2).flatten(1))


class GlobalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.fc = nn.Linear(4, 2)

    def features(self, x):
        return F.relu(self.conv(x))

    def forward(self, x):
        return self.fc(self.features(x).amax(dim=(2, 3)))    # one number per feature map


def shift_test(model, x, shift):
    rolled = torch.roll(x, shifts=shift, dims=3)
    with torch.no_grad():
        f_orig, f_rolled = model.features(x), model.features(rolled)
        inner = slice(2, 10)
        conv_shifts = torch.allclose(f_rolled[..., inner], torch.roll(f_orig, shift, dims=3)[..., inner], atol=1e-5)
        output_same = torch.allclose(model(x), model(rolled), atol=1e-5)
    return conv_shifts, output_same

**Why it works.** A convolution uses the same kernel at every position, so shifting the input shifts the feature maps by the same amount: convolution is **translation-equivariant**. That alone does not make the classifier ignore position. `TinyCNN` flattens its pooled maps and gives every position its own weights in the linear layer, so the same feature at a new place reaches the output through different, possibly untrained, weights. Global max-pooling keeps only the strongest response of each map, wherever it occurs, which makes the output **translation-invariant** (up to effects at the borders).

### Part 3: Which networks recognize bars in new places?

In [ ]:
def train(model, X, y, steps=100, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss.item()


def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()


def compare(builders, seeds=5):
    """name -> (accuracies on X_seen, accuracies on X_new), one entry per seed."""
    results = {}
    for name, build in builders.items():
        seen, new = [], []
        for seed in range(seeds):
            torch.manual_seed(seed)
            model = build()
            train(model, X_train, y_train)
            seen.append(accuracy(model, X_seen, y_seen))
            new.append(accuracy(model, X_new, y_new))
        results[name] = (seen, new)
    return results

**Why it works.** All three networks fit the training positions perfectly, so training accuracy says nothing about which one learned "vertical versus horizontal". Only `GlobalCNN`, with 50 parameters, recognizes bars at new positions every time. The dense network does worse than guessing: a vertical bar in column 8 lights every row of that column, and its rows 2 to 6 are exactly the pixels that every training horizontal bar lit, so it is called horizontal, and vice versa. It learned "where the bright pixels are", not "which way they run". `TinyCNN` has the right kind of feature detector, but its linear layer gives every position its own weights, and the responses at positions 7 to 9 reach the output through weights that were trained on other evidence. It scores about 0.24, not much better than the dense network. The design lesson: convolution provides equivariance, and pooling over all positions turns it into invariance, which is what a "is there a bar anywhere" label needs.